# 03 — Evaluación de los modelos V4 + V5

Carga los artefactos producidos por `02_modelo_fraude.ipynb` y reporta:

- AUC-ROC mean ± std (5-fold CV)
- Matriz de confusión en el holdout 20%
- Curva ROC + curva de calibración
- SHAP summary plot (importancia + dirección por feature)
- Distribución del score de anomalía por tier

Si algo aquí falla, vuelve a correr el cuaderno 02 — los artefactos pueden estar desactualizados respecto al dataset.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "app").is_dir() and (REPO_ROOT.parent / "app").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split

from notebooks._training import (
    ANOMALY_MODEL_PATH,
    FRAUD_MODEL_PATH,
    build_dataset,
)

%matplotlib inline

In [ ]:
dataset = build_dataset()
print(
    f"rows = {dataset.X.shape[0]}, features = {dataset.X.shape[1]}, "
    f"positive_rate = {dataset.positive_rate:.3f}"
)

booster = lgb.Booster(model_file=str(FRAUD_MODEL_PATH))
iforest = joblib.load(str(ANOMALY_MODEL_PATH))
print("artefactos cargados:")
print(f"  - {FRAUD_MODEL_PATH}")
print(f"  - {ANOMALY_MODEL_PATH}")

## AUC en holdout + 5-fold CV

In [ ]:
X, y = dataset.X, dataset.y
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

y_holdout_pred = booster.predict(X_holdout)
holdout_auc = roc_auc_score(y_holdout, y_holdout_pred)
print(f"Holdout AUC-ROC : {holdout_auc:.3f}")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []
for _tr, va in skf.split(X_dev, y_dev):
    fold_aucs.append(roc_auc_score(y_dev[va], booster.predict(X_dev[va])))
print(f"CV AUC-ROC      : {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}")
print("fold AUCs       :", [round(a, 3) for a in fold_aucs])

## Matriz de confusión + classification report

In [ ]:
threshold = 0.5
y_pred = (y_holdout_pred >= threshold).astype(int)
cm = confusion_matrix(y_holdout, y_pred)
print("confusion matrix @ threshold=0.5")
print(pd.DataFrame(cm, index=["verdad=0", "verdad=1"], columns=["pred=0", "pred=1"]))
print()
print(classification_report(y_holdout, y_pred, target_names=["normal", "posible_fraude"]))

## Curva ROC + calibración

In [ ]:
fpr, tpr, _ = roc_curve(y_holdout, y_holdout_pred)
frac_pos, mean_pred = calibration_curve(y_holdout, y_holdout_pred, n_bins=10)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(fpr, tpr, label=f"AUC={holdout_auc:.3f}")
axes[0].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC — holdout")
axes[0].legend(loc="lower right")

axes[1].plot(mean_pred, frac_pos, marker="o")
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[1].set_xlabel("probabilidad media predicha")
axes[1].set_ylabel("fracción positivos")
axes[1].set_title("Calibración")

plt.tight_layout()
plt.show()

## SHAP summary

In [ ]:
import shap

explainer = shap.TreeExplainer(booster)
shap_values = explainer.shap_values(X_holdout)
if isinstance(shap_values, list):
    shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]

shap.summary_plot(
    shap_values,
    pd.DataFrame(X_holdout, columns=dataset.feature_names),
    plot_type="dot",
    show=True,
)

## Distribución del anomaly score por etiqueta

In [ ]:
anomaly_scores = iforest.score_samples(pd.DataFrame(X, columns=dataset.feature_names))
frame = pd.DataFrame({"anomaly_score": anomaly_scores, "label": y})

fig, ax = plt.subplots(figsize=(8, 4))
frame[frame.label == 0]["anomaly_score"].hist(bins=30, alpha=0.6, label="normal", ax=ax)
frame[frame.label == 1]["anomaly_score"].hist(bins=30, alpha=0.6, label="posible_fraude", ax=ax)
ax.set_xlabel("anomaly score (más bajo = más anómalo)")
ax.set_ylabel("frecuencia")
ax.set_title("Distribución de score de anomalía")
ax.legend()
plt.tight_layout()
plt.show()